<a href="https://colab.research.google.com/github/nexageapps/LLM/blob/main/01_Basic/L8_Tokenizers_Deep_Dive.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# L8: Tokenizers Deep Dive - Breaking Text into Tokens

**Author:** Karthik Arjun  
**LinkedIn:** [Karthik Arjun](https://www.linkedin.com/in/karthik-arjun-a5b4a258/)  
**Level:** Basic  
**Lesson:** 8 of 15

---

## Learning Objectives

By the end of this lesson, you will:
- Understand why tokenization is crucial for LLMs
- Learn how Byte Pair Encoding (BPE) works
- Understand WordPiece tokenization used in BERT
- Explore SentencePiece and its advantages
- Compare different tokenization algorithms
- Train custom tokenizers for specific domains

---

## Concept: Tokenization - The Bridge Between Text and Models

**Tokenization** is the process of breaking text into smaller units (tokens) that language models can process.

### Why Tokenization?

**The Challenge:**
- Models can't process raw text directly
- Need to convert text to numbers
- Balance between vocabulary size and representation quality

**The Solution:**
- Break text into subword units
- Create a fixed vocabulary
- Map tokens to numerical IDs

### Three Main Tokenization Algorithms

1. **Byte Pair Encoding (BPE)**
   - Merges most frequent character pairs iteratively
   - Used by: GPT-2, GPT-3, RoBERTa
   - Bottom-up approach

2. **WordPiece**
   - Similar to BPE but uses likelihood-based merging
   - Used by: BERT, DistilBERT
   - Maximizes language model likelihood

3. **SentencePiece**
   - Language-agnostic, treats text as raw bytes
   - Used by: T5, ALBERT, XLNet
   - No pre-tokenization required

---

In [ ]:
# Step 1: Install and Import Required Libraries
!pip install transformers tokenizers sentencepiece matplotlib numpy -q

import re
from collections import defaultdict, Counter
from transformers import AutoTokenizer
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders, processors
import sentencepiece as spm
import matplotlib.pyplot as plt
import numpy as np

print("Libraries imported successfully!")
print(f"Transformers tokenizers ready!")

## Part 1: Byte Pair Encoding (BPE)

**BPE** is a data compression algorithm adapted for tokenization. It starts with characters and iteratively merges the most frequent pairs.

### How BPE Works:

1. **Initialize:** Start with all characters as tokens
2. **Count Pairs:** Find most frequent adjacent token pair
3. **Merge:** Replace all occurrences with new token
4. **Repeat:** Continue until desired vocabulary size

### Example:

```
Text: "low low low lower lowest"

Iteration 0: ['l', 'o', 'w', ' ', 'l', 'o', 'w', ...]
Most frequent pair: ('l', 'o') -> merge to 'lo'

Iteration 1: ['lo', 'w', ' ', 'lo', 'w', ...]
Most frequent pair: ('lo', 'w') -> merge to 'low'

Iteration 2: ['low', ' ', 'low', ' ', 'low', ...]
```

### Advantages:
- Handles rare words through subwords
- No unknown tokens (can represent any text)
- Efficient vocabulary size

---

In [ ]:
# Step 2: Implement Simple BPE from Scratch

def get_stats(vocab):
    """
    Count frequency of adjacent token pairs.
    """
    pairs = defaultdict(int)
    for word, freq in vocab.items():
        symbols = word.split()
        for i in range(len(symbols) - 1):
            pairs[symbols[i], symbols[i + 1]] += freq
    return pairs

def merge_vocab(pair, vocab):
    """
    Merge all occurrences of the most frequent pair.
    """
    new_vocab = {}
    bigram = ' '.join(pair)
    replacement = ''.join(pair)
    
    for word in vocab:
        new_word = word.replace(bigram, replacement)
        new_vocab[new_word] = vocab[word]
    
    return new_vocab

def train_bpe(text, num_merges):
    """
    Train BPE tokenizer on text.
    """
    # Initialize vocabulary with character-level tokens
    vocab = {}
    for word in text.split():
        # Add space between characters
        word_chars = ' '.join(list(word)) + ' </w>'
        vocab[word_chars] = vocab.get(word_chars, 0) + 1
    
    print("Initial vocabulary (first 5 words):")
    for i, (word, freq) in enumerate(list(vocab.items())[:5]):
        print(f"  {word} : {freq}")
    print()
    
    # Perform merges
    merges = []
    for i in range(num_merges):
        pairs = get_stats(vocab)
        if not pairs:
            break
        
        best_pair = max(pairs, key=pairs.get)
        vocab = merge_vocab(best_pair, vocab)
        merges.append(best_pair)
        
        if i < 5:  # Show first 5 merges
            print(f"Merge {i+1}: {best_pair[0]} + {best_pair[1]} -> {''.join(best_pair)} (freq: {pairs[best_pair]})")
    
    return vocab, merges

# Train BPE on sample text
sample_text = "low low low low lower lower newest newest newest newest widest widest widest"

print("Training BPE Tokenizer\n")
print(f"Sample text: {sample_text}\n")

final_vocab, merge_operations = train_bpe(sample_text, num_merges=10)

print("\nFinal vocabulary (first 10 tokens):")
for i, (word, freq) in enumerate(list(final_vocab.items())[:10]):
    print(f"  {word} : {freq}")

print(f"\nTotal merge operations: {len(merge_operations)}")
print("\nBPE learns to merge frequent character sequences into subword units!")

## Part 2: BPE with HuggingFace Tokenizers

Let's use a real BPE tokenizer from GPT-2 and see how it handles different texts.

### GPT-2 Tokenizer:
- Vocabulary size: ~50,000 tokens
- Uses BPE algorithm
- Handles byte-level encoding

---

In [ ]:
# Step 3: Explore GPT-2 BPE Tokenizer

print("Loading GPT-2 BPE Tokenizer\n")
tokenizer_gpt2 = AutoTokenizer.from_pretrained("gpt2")

# Test sentences
test_sentences = [
    "Hello, how are you?",
    "Tokenization is fundamental.",
    "supercalifragilisticexpialidocious",
    "COVID-19 pandemic",
    "machine learning"
]

print("BPE Tokenization Examples:\n")

for sent in test_sentences:
    # Tokenize
    tokens = tokenizer_gpt2.tokenize(sent)
    token_ids = tokenizer_gpt2.encode(sent)
    
    print(f"Text: {sent}")
    print(f"Tokens: {tokens}")
    print(f"Token IDs: {token_ids}")
    print(f"Number of tokens: {len(tokens)}")
    print()

print("Notice how BPE handles:")
print("  - Common words: Single tokens")
print("  - Rare words: Multiple subword tokens")
print("  - Unknown words: Character-level fallback")

## Part 3: WordPiece Tokenization

**WordPiece** is similar to BPE but uses a different merging criterion. Instead of frequency, it maximizes the likelihood of the training data.

### How WordPiece Differs from BPE:

**BPE:**
- Merges most frequent pair
- Frequency-based

**WordPiece:**
- Merges pair that maximizes likelihood
- Likelihood-based: P(pair) / (P(first) * P(second))

### BERT's WordPiece:
- Uses ## prefix for subword continuations
- Example: "playing" -> ["play", "##ing"]
- Vocabulary size: ~30,000 tokens

---

In [ ]:
# Step 4: Explore BERT WordPiece Tokenizer

print("Loading BERT WordPiece Tokenizer\n")
tokenizer_bert = AutoTokenizer.from_pretrained("bert-base-uncased")

# Test sentences
test_sentences = [
    "Hello, how are you?",
    "Tokenization is fundamental.",
    "supercalifragilisticexpialidocious",
    "COVID-19 pandemic",
    "playing, played, player"
]

print("WordPiece Tokenization Examples:\n")

for sent in test_sentences:
    # Tokenize
    tokens = tokenizer_bert.tokenize(sent)
    token_ids = tokenizer_bert.encode(sent, add_special_tokens=False)
    
    print(f"Text: {sent}")
    print(f"Tokens: {tokens}")
    print(f"Token IDs: {token_ids}")
    print(f"Number of tokens: {len(tokens)}")
    print()

print("Notice the ## prefix for subword continuations!")
print("Example: 'playing' -> ['play', '##ing']")

## Part 4: SentencePiece Tokenization

**SentencePiece** is a language-agnostic tokenizer that treats text as a sequence of Unicode characters.

### Key Features:

1. **No Pre-tokenization:** Doesn't require word boundaries
2. **Language-agnostic:** Works for any language (including Chinese, Japanese)
3. **Reversible:** Can perfectly reconstruct original text
4. **Whitespace as token:** Treats spaces as regular characters

### Advantages:
- Works for languages without spaces
- No language-specific preprocessing
- Used in multilingual models (T5, mT5, ALBERT)

---

In [ ]:
# Step 5: Explore T5 SentencePiece Tokenizer

print("Loading T5 SentencePiece Tokenizer\n")
tokenizer_t5 = AutoTokenizer.from_pretrained("t5-small")

# Test sentences (including non-English)
test_sentences = [
    "Hello, how are you?",
    "Tokenization is fundamental.",
    "machine learning",
    "COVID-19 pandemic"
]

print("SentencePiece Tokenization Examples:\n")

for sent in test_sentences:
    # Tokenize
    tokens = tokenizer_t5.tokenize(sent)
    token_ids = tokenizer_t5.encode(sent, add_special_tokens=False)
    
    print(f"Text: {sent}")
    print(f"Tokens: {tokens}")
    print(f"Token IDs: {token_ids}")
    print(f"Number of tokens: {len(tokens)}")
    print()

print("Notice the underscore (_) representing spaces!")
print("SentencePiece treats whitespace as a regular token.")

## Part 5: Comparing Tokenizers

Let's compare how different tokenizers handle the same text.

---

In [ ]:
# Step 6: Side-by-Side Comparison

comparison_text = "The transformer architecture revolutionized natural language processing."

print("Tokenizer Comparison\n")
print(f"Text: {comparison_text}\n")

# GPT-2 (BPE)
tokens_gpt2 = tokenizer_gpt2.tokenize(comparison_text)
print(f"GPT-2 (BPE):")
print(f"  Tokens: {tokens_gpt2}")
print(f"  Count: {len(tokens_gpt2)}")
print()

# BERT (WordPiece)
tokens_bert = tokenizer_bert.tokenize(comparison_text)
print(f"BERT (WordPiece):")
print(f"  Tokens: {tokens_bert}")
print(f"  Count: {len(tokens_bert)}")
print()

# T5 (SentencePiece)
tokens_t5 = tokenizer_t5.tokenize(comparison_text)
print(f"T5 (SentencePiece):")
print(f"  Tokens: {tokens_t5}")
print(f"  Count: {len(tokens_t5)}")
print()

# Visualize comparison
fig, ax = plt.subplots(figsize=(10, 6))

tokenizer_names = ['GPT-2\n(BPE)', 'BERT\n(WordPiece)', 'T5\n(SentencePiece)']
token_counts = [len(tokens_gpt2), len(tokens_bert), len(tokens_t5)]
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1']

bars = ax.bar(tokenizer_names, token_counts, color=colors, edgecolor='black', linewidth=2)

# Add value labels on bars
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{int(height)}',
            ha='center', va='bottom', fontsize=12, fontweight='bold')

ax.set_ylabel('Number of Tokens', fontsize=12)
ax.set_title('Tokenizer Comparison: Same Text, Different Token Counts', fontsize=14, fontweight='bold')
ax.set_ylim(0, max(token_counts) * 1.2)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print("Different tokenizers produce different token counts for the same text!")

## Part 6: Training a Custom Tokenizer

Let's train a custom BPE tokenizer on domain-specific text.

---

In [ ]:
# Step 7: Train Custom BPE Tokenizer

from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

# Create sample domain-specific corpus (medical text)
medical_corpus = """
The patient presented with acute myocardial infarction.
Diagnosis: hypertension and diabetes mellitus type 2.
Prescription: metformin 500mg twice daily.
The cardiovascular system includes the heart and blood vessels.
Symptoms include chest pain, dyspnea, and palpitations.
Treatment involves medication, lifestyle changes, and monitoring.
The patient's electrocardiogram showed ST-segment elevation.
Laboratory tests revealed elevated troponin levels.
The physician recommended immediate cardiac catheterization.
Post-operative care includes anticoagulation therapy.
"""

# Save corpus to file
with open('medical_corpus.txt', 'w') as f:
    f.write(medical_corpus)

print("Training Custom BPE Tokenizer on Medical Text\n")

# Initialize tokenizer
tokenizer_custom = Tokenizer(BPE(unk_token="[UNK]"))
tokenizer_custom.pre_tokenizer = Whitespace()

# Configure trainer
trainer = BpeTrainer(
    vocab_size=500,
    special_tokens=["[UNK]", "[CLS]", "[SEP]", "[PAD]", "[MASK]"]
)

# Train tokenizer
tokenizer_custom.train(files=['medical_corpus.txt'], trainer=trainer)

print("Custom tokenizer trained!\n")

# Test custom tokenizer
test_medical_texts = [
    "The patient has hypertension.",
    "Myocardial infarction requires immediate treatment.",
    "Electrocardiogram shows abnormalities."
]

print("Custom Tokenizer Results:\n")

for text in test_medical_texts:
    output = tokenizer_custom.encode(text)
    print(f"Text: {text}")
    print(f"Tokens: {output.tokens}")
    print(f"IDs: {output.ids}")
    print()

print("The custom tokenizer learned medical terminology as single tokens!")

## Part 7: Tokenizer Properties and Trade-offs

Let's analyze important tokenizer properties.

---

In [ ]:
# Step 8: Analyze Tokenizer Properties

def analyze_tokenizer(tokenizer, name, test_texts):
    """
    Analyze tokenizer properties.
    """
    print(f"\n{name} Analysis:")
    print(f"  Vocabulary size: {tokenizer.vocab_size:,}")
    
    total_tokens = 0
    total_chars = 0
    
    for text in test_texts:
        tokens = tokenizer.tokenize(text)
        total_tokens += len(tokens)
        total_chars += len(text)
    
    compression_ratio = total_chars / total_tokens
    print(f"  Average compression ratio: {compression_ratio:.2f} chars/token")
    
    # Test rare word handling
    rare_word = "pneumonoultramicroscopicsilicovolcanoconiosis"
    rare_tokens = tokenizer.tokenize(rare_word)
    print(f"  Rare word '{rare_word}':")
    print(f"    Tokens: {rare_tokens}")
    print(f"    Count: {len(rare_tokens)}")

# Test corpus
test_corpus = [
    "Machine learning is transforming technology.",
    "Natural language processing enables computers to understand text.",
    "Deep learning models require large datasets.",
    "Artificial intelligence is advancing rapidly."
]

print("Tokenizer Property Analysis")
print("=" * 50)

analyze_tokenizer(tokenizer_gpt2, "GPT-2 (BPE)", test_corpus)
analyze_tokenizer(tokenizer_bert, "BERT (WordPiece)", test_corpus)
analyze_tokenizer(tokenizer_t5, "T5 (SentencePiece)", test_corpus)

print("\n" + "=" * 50)
print("\nKey Insights:")
print("  - Larger vocabulary = fewer tokens per text")
print("  - Subword tokenization handles rare words gracefully")
print("  - Different algorithms optimize for different goals")

## Exercises

### Exercise 1: BPE from Scratch
Implement BPE on your own text corpus:
```python
my_text = "your custom text here repeated multiple times"
vocab, merges = train_bpe(my_text, num_merges=20)
# Analyze the learned merges
```

### Exercise 2: Compare Tokenizers on Different Languages
Test GPT-2, BERT, and T5 tokenizers on:
- English text
- Text with numbers and special characters
- Code snippets
- Mixed-language text

Which tokenizer handles each case best?

### Exercise 3: Vocabulary Size Impact
Train custom tokenizers with different vocabulary sizes (100, 500, 1000, 5000):
- How does vocabulary size affect token count?
- What's the trade-off between vocabulary size and coverage?

### Exercise 4: Domain-Specific Tokenizer
Train a custom tokenizer for a specific domain:
- Collect domain-specific text (legal, medical, code, etc.)
- Train BPE or WordPiece tokenizer
- Compare with general-purpose tokenizers
- Measure improvement in token efficiency

### Exercise 5: Tokenization Efficiency
Analyze tokenization efficiency:
```python
import time

text = "your test text" * 1000

start = time.time()
tokens = tokenizer.tokenize(text)
end = time.time()

print(f"Time: {end - start:.4f}s")
print(f"Tokens/second: {len(tokens) / (end - start):.0f}")
```

Compare speed across different tokenizers.

---

## Key Takeaways

1. **Tokenization** converts text into subword units that models can process
2. **BPE (Byte Pair Encoding)** merges frequent character pairs iteratively
3. **WordPiece** uses likelihood-based merging instead of frequency
4. **SentencePiece** is language-agnostic and treats text as raw bytes
5. **Subword tokenization** handles rare words without unknown tokens
6. **Vocabulary size** is a trade-off between coverage and efficiency
7. **Custom tokenizers** can be trained for domain-specific applications
8. **Different tokenizers** produce different token counts for the same text

### Tokenizer Selection Guide:

**Use BPE (GPT-2) when:**
- Building generative models
- Need byte-level encoding
- Working with English text

**Use WordPiece (BERT) when:**
- Building understanding models
- Need likelihood-based merging
- Working with classification tasks

**Use SentencePiece (T5) when:**
- Working with multiple languages
- Need language-agnostic tokenization
- Building multilingual models

---

## Additional Resources

### Papers
- **"Neural Machine Translation of Rare Words with Subword Units"** (Sennrich et al., 2016) - BPE
- **"Japanese and Korean Voice Search"** (Schuster & Nakajima, 2012) - WordPiece
- **"SentencePiece: A simple and language independent approach"** (Kudo & Richardson, 2018)

### Tutorials
- [HuggingFace Tokenizers Documentation](https://huggingface.co/docs/tokenizers/)
- [The Illustrated Word2vec](http://jalammar.github.io/illustrated-word2vec/)
- [SentencePiece GitHub](https://github.com/google/sentencepiece)

### Interactive
- [Tokenizer Playground](https://huggingface.co/spaces/Xenova/the-tokenizer-playground)
- [BPE Visualizer](https://github.com/karpathy/minbpe)

---

## Next Lesson

**L9: Model Loading and Inference** - Learn how to load models from HuggingFace Hub and perform inference

---

**Congratulations! You now understand how tokenizers work!**

*You've learned the foundation of text processing: how LLMs break down text into tokens they can understand.*